# Completion Basics

---

## Use case

La **completion** est la primitive fondamentale des LLMs : envoyer un prompt, recevoir une réponse.

Ce notebook couvre les mécaniques essentielles :
- Client synchrone et asynchrone
- Paramètres clés (temperature, max_tokens)
- Streaming
- Limites fondamentales du modèle (mémoire, knowledge cutoff)
- Transition vers AI SDK (Vercel) pour une approche provider-agnostic

## Stack

- **OpenAI SDK** (`npm:openai`) — client officiel, utilisé ici pour la pédagogie
- **AI SDK Vercel** (`npm:ai`) — abstraction provider-agnostic TypeScript-first


## Setup

**En local**
1. Copier `.env.example` en `.env` à la racine du repo
2. Renseigner `OPENAI_API_KEY`
3. Lancer le notebook avec le kernel Deno

**Sur Google Colab**
> ⚠️ Le kernel Deno n'est pas disponible nativement sur Colab. Ce notebook est prévu pour un environnement local.

In [2]:
import { load } from "jsr:@std/dotenv";

const env = await load({ envPath: "../../.env" });
const apiKey = env["OPENAI_API_KEY"] ?? Deno.env.get("OPENAI_API_KEY");

if (!apiKey) throw new Error("OPENAI_API_KEY manquante — voir .env.example");

## Client OpenAI

L'API OpenAI s'articule autour de deux concepts simples :
- **system prompt**: le comportement global du modèle
- **user prompt**: le message de l'utilisateur

La réponse contient la completion mais aussi des métadonnées utiles : usage (tokens consommés), model utilisé, etc.

In [ ]:
import OpenAI from "npm:openai";

const client = new OpenAI({ apiKey });

const response = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: [
    { role: "system", content: "You are a helpful assistant." },
    { role: "user", content: "What is a Python tuple? Answer in one sentence." },
  ],
});

console.log(response.choices[0].message.content);

In [ ]:
// Tokens consommés — utile pour estimer les coûts
console.log("Usage:", response.usage);

// Model effectivement utilisé
console.log("Model:", response.model);

// Output complet
console.log("Choices:", response.choices);

## Paramètres clés

Trois paramètres influencent directement la qualité et le coût des completions :

- **`temperature`** — créativité de la réponse. `0` = déterministe, `1` = plus créatif. Par défaut `1`.
- **`max_completion_tokens`** — nombre maximum de tokens générés. Utile pour maîtriser les coûts et forcer la concision.
- **`top_p`** — alternative à `temperature` pour contrôler la diversité. En pratique, on ajuste l'un ou l'autre, pas les deux.

> **Règle empirique** — tâches factuelles/code → `temperature` basse (0–0.3). Génération créative → `temperature` haute (0.7–1).

### Temperature

In [ ]:
const prompt = "Give me a one-sentence tagline for a coffee shop.";

for (const temperature of [0, 0.7, 1.5]) {
  const response = await client.chat.completions.create({
    model: "gpt-4.1-mini",
    messages: [{ role: "user", content: prompt }],
    temperature,
  });
  console.log(`temperature=${temperature} → ${response.choices[0].message.content}`);
}

### Max tokens

In [ ]:
const response = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: [{ role: "user", content: "Explain how the internet works." }],
  max_completion_tokens: 50,
});

console.log(response.choices[0].message.content);
console.log(`\nTokens — prompt: ${response.usage?.prompt_tokens}, completion: ${response.usage?.completion_tokens}`);

## Streaming

Par défaut, la réponse est renvoyée en une seule fois une fois la génération terminée.
Le streaming envoie les tokens au fur et à mesure, ce qui change deux choses :

- **UX**: l'utilisateur voit la réponse apparaître progressivement, sans attente perçue
- **TTFB** (Time To First Byte): le premier token arrive bien avant la fin de la génération

Indispensable en production pour toute interface conversationnelle.

In [ ]:
const stream = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: [{ role: "user", content: "Explain how the internet works in a few sentences." }],
  stream: true,
});

let text = "";

for await (const chunk of stream) {
  const delta = chunk.choices[0]?.delta?.content ?? "";
  if (delta) console.log(delta);
}

## Limites fondamentales

Deux limites importantes à comprendre avant d'aller plus loin :

**Pas de mémoire** — chaque appel est indépendant. Le modèle ne se souvient pas des échanges précédents. C'est au développeur de gérer le contexte conversationnel en passant l'historique explicitement à chaque appel. → `02-augmentation/memoire`

**Knowledge cutoff** — le modèle ne connaît que les données sur lesquelles il a été entraîné. Il ne sait rien des événements récents, et rien de vos données internes. → `02-augmentation/rag`

### Mémoire

In [ ]:
const response1 = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: [{ role: "user", content: "Hi! My name is Laurent." }],
});
console.log("Response 1:", response1.choices[0].message.content);

// Nouvel appel — le modèle n'a aucun souvenir de l'échange précédent
const response2 = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: [{ role: "user", content: "What is my name?" }],
});
console.log("Response 2:", response2.choices[0].message.content);

# Knowledge cutoff

In [ ]:
// Événement récent — le modèle ne sait pas
const response1 = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: [{ role: "user", content: "How many medals did France win at the 2026 Winter Olympics?" }],
});
console.log("Événement récent:", response1.choices[0].message.content);

// Donnée interne — le modèle ne sait pas non plus
const response2 = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: [{ role: "user", content: "How do I request time off at Younup?" }],
});
console.log("Donnée interne:", response2.choices[0].message.content);

## Vers une approche provider-agnostic avec AI SDK

Jusqu'ici on a utilisé le SDK OpenAI directement, idéal pour comprendre la mécanique.

**AI SDK (Vercel)** expose une interface unifiée TypeScript-first pour switcher de provider sans réécrire le code. Le format est cohérent quel que soit le provider.

Les providers sont des packages séparés :
- `npm:@ai-sdk/openai`
- `npm:@ai-sdk/anthropic`
- `npm:@ai-sdk/mistral`

In [3]:
// Changer ici pour switcher de provider
import { createOpenAI } from "npm:@ai-sdk/openai";
// import { createAnthropic } from "npm:@ai-sdk/anthropic";

const openai = createOpenAI({ apiKey });
const MODEL = openai("gpt-4.1-mini");

// Anthropic — décommenter et renseigner ANTHROPIC_API_KEY dans .env
// const anthropic = createAnthropic({ apiKey: env["ANTHROPIC_API_KEY"] });
// const MODEL = anthropic("claude-haiku-4-5");

In [7]:
import { generateText } from "npm:ai";

const { text, usage } = await generateText({
  model: MODEL,
  system: "You are a helpful assistant.",
  prompt: "What is a Python tuple? Answer in one sentence.",
});

console.log(text);
console.log(`\nTokens — prompt: ${usage.inputTokens}, completion: ${usage.outputTokens}`);

A Python tuple is an immutable, ordered collection of elements enclosed in parentheses.

Tokens — prompt: 28, completion: 16
